In [ ]:
# KNN 
import numpy as np
import random as rd
import math
from sklearn import datasets
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

#=====================================
class Escalador:
    def __init__(self,u=None,sigma=None):
        self.u = u
        self.sigma = sigma
    #--------------------------
    def escalar_train(self,X):
        X = np.array(X)
        X_es = X.copy()
        u = X_es.mean(axis=0)
        sigma = X_es.std(axis=0)
        X_es =( X_es - u )/sigma
        self.u = u
        self.sigma = sigma
        return X_es
    #---------------------------------
    def escalar_test(self,X):
        X_test = X.copy()
        X_test = (X_test -self.u)/self.sigma
        return X_test
#=====================================
class Splitter: 
    #---------------------------------------
    def permutacion(self,n,n_sub):                 # PC1: l 8 columnas , n_sub 5 predictores para EFS mlxtend 
        f = []
        k = n
        l = list(range(n))
        while k>1:
            i = math.floor(k*rd.uniform(0,1)) 
            l[k-1] , l[i] = l[i] , l[k-1]
            f.append(l[k-1])
            if (len(l) - k) + 1  == n_sub:
                break
            k = k - 1 
        return f
    #----------------------------------------
    def dividir_datos(self,X,y,por):
        n = len(X)
        n_sub = n - int(n*por/100)
        indices = list(range(n))
        inds =  self.permutacion(n,n_sub)
        resto = list(set(indices) - set(inds))
        X_train,X_test,y_train,y_test = X[inds],X[resto],y[inds],y[resto]
        return  X_train,X_test,y_train,y_test
    #---------------------------------------- 
    
#================================
class KNN:
    def __init__(self,metrica):  #1,2
        self.metrica =  metrica
        self.X_train = None
        self.y_train = None
    #------------------------------
    def distancia(self,a,b):
        if self.metrica == 2:
            d = 0
            for i in range(len(a)):
                d +=(a[i]-b[i])**2
            return d**.5
        if self.metrica == 1:
            d = 0
            for i in range(len(a)):
                d+=abs(a[i] - b[i])
            return d
    #-----------------------------
    def ordenar(self,lista):
        n = len(lista)   # [(float,int)]
        for i in range(1,n):
            var = lista[i][0]
            fila =lista[i]
            j = i - 1
            while j>=0 and lista[j][0]>var:
                lista[j + 1] = lista[j]
                j = j - 1
            lista[j + 1] = fila
        return lista
    #---------------------------
    def entrenar(self,X,y):
        self.X_train = X
        self.y_train = y 
    #------------------------------
    def votar(self,distancias,k): 
        d_s = []
        vecinos = []
        for i in range(k):
            d_s.append(distancias[i][0])
            vecinos.append(distancias[i][1]) 
        return d_s, vecinos 
    #------------------------------
    def conteo(self,clases):
        conteo = {}
        for c in clases:
            if c not in conteo:
                conteo[c] = 0
            conteo[c] +=1
        clase = max(conteo,key = conteo.get)
        return clase
    #--------------------------------
    def predecir(self,X_test,k):
        predicciones = []
        for i in range(len(X_test)):
            distancias = []
            for j in range(len(self.X_train)): 
                d = self.distancia(X_test[i],self.X_train[j])
                distancias.append((d,j))
            distancias = self.ordenar(distancias)
            _,vecinos = self.votar(distancias,k)
            clases = self.y_train[vecinos]
            clase = self.conteo(clases)
            predicciones.append(clase) # (array,float)
        return predicciones
    #----------------------------------
     
    #======================================
if __name__=='__main__':
    iris = datasets.load_iris()
    X = iris.data[:,:2]
    y = iris.target
    separador = Splitter()
    escalador = Escalador()
    X_train,X_test,y_train,y_test = separador.dividir_datos(X,y,30)
    X_train = escalador.escalar_train(X_train)
    X_test = escalador.escalar_test(X_test)
    knn = KNN(2)
    knn.entrenar(X_train,y_train)
    y_pred = knn.predecir(X_test,3)
    print(f"Precisión del modelo (Accuracy): {accuracy_score(y_test, y_pred):.2f}")
    print(f"train: {len(X_train)}  test: {len(X_test)}")
    print(f"total: {len(X_train)+len(X_test)}  total iris: {len(X)}")
    print(classification_report(y_test, y_pred, target_names=iris.target_names))
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",xticklabels=iris.target_names, yticklabels=iris.target_names)
    plt.xlabel('Predicción')
    plt.ylabel('Realidad')
    plt.title(f'Matriz de Confusión (K={k})')
    plt.show()
    X_set, y_set = X_train, y_train 
    x_min, x_max = X_set[:, 0].min() - 1, X_set[:, 0].max() + 1
    y_min, y_max = X_set[:, 1].min() - 1, X_set[:, 1].max() + 1 
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01),np.arange(y_min, y_max, 0.01))
    Z = knn.predecir(np.c_[xx.ravel(), yy.ravel()],3)
    Z = np.array(Z)
    Z = Z.reshape(xx.shape)
    plt.figure(figsize=(10, 6))
    cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF']) 
    cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])
    plt.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.4) 
    sns.scatterplot(x=X_set[:, 0], y=X_set[:, 1], hue=iris.target_names[y_set],palette=['red', 'green', 'blue'], edgecolor="black", s=100)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    plt.title(f"Clasificación KNN (k={k}) - Fronteras de Decisión")
    plt.xlabel('Sépalo Largo (Estandarizado)')
    plt.ylabel('Sépalo Ancho (Estandarizado)')
    plt.legend()
    plt.show()

In [ ]:
# SKLEARN
#SKLEARN
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

# Importamos módulos de scikit-learn
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ==========================================
# 1. PREPARACIÓN DE DATOS
# ==========================================
# Cargamos el dataset Iris
iris = datasets.load_iris()

# Para poder visualizarlo en 2D, tomamos solo las 2 primeras características:
# (Sepal Length, Sepal Width).
# NOTA: En un caso real usarías todas las características.
X = iris.data[:, :2]  
y = iris.target

# Dividimos en conjunto de Entrenamiento (70%) y Prueba (30%)
# random_state=42 asegura que el resultado sea reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# ==========================================
# 2. ESCALADO DE DATOS (CRUCIAL PARA KNN)
# ==========================================
# Como KNN usa distancias, si una variable tiene números grandes (ej. 1000) 
# y otra pequeños (ej. 0.1), dominará el cálculo. Estandarizamos para evitarlo.
scaler = StandardScaler()

# Ajustamos el escalador solo con los datos de entrenamiento para evitar "data leakage"
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================================
# 3. ENTRENAMIENTO DEL MODELO
# ==========================================
k = 5  # Probamos con 5 vecinos
# metric='minkowski', p=2 equivale a Distancia Euclidiana estándar
knn = KNeighborsClassifier(n_neighbors=k, metric='minkowski', p=2)

knn.fit(X_train, y_train)

# ==========================================
# 4. PREDICCIÓN Y EVALUACIÓN
# ==========================================
y_pred = knn.predict(X_test)

print(f"Precisión del modelo (Accuracy): {accuracy_score(y_test, y_pred):.2f}")
print("\n--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

print("\n--- Matriz de Confusión ---")
# Visualizamos la matriz con Seaborn para que se vea bonita
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel('Predicción')
plt.ylabel('Realidad')
plt.title(f'Matriz de Confusión (K={k})')
plt.show()

# ==========================================
# 5. VISUALIZACIÓN DE FRONTERAS (MAGIA VISUAL)
# ==========================================
# Creamos una malla (grid) para colorear el fondo según la predicción del modelo
X_set, y_set = X_train, y_train

# Definimos los límites del gráfico
x_min, x_max = X_set[:, 0].min() - 1, X_set[:, 0].max() + 1
y_min, y_max = X_set[:, 1].min() - 1, X_set[:, 1].max() + 1

# Generamos la malla con pasos de 0.01
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01),
                     np.arange(y_min, y_max, 0.01))

# Predecimos la clase para CADA punto de la malla (fondo)
Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Graficamos
plt.figure(figsize=(10, 6))
# Colores para las zonas: Rojo, Verde, Azul (claros)
cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
# Colores para los puntos: Rojo, Verde, Azul (fuertes)
cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])

plt.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.4)

# Graficamos los puntos de entrenamiento encima
sns.scatterplot(x=X_set[:, 0], y=X_set[:, 1], hue=iris.target_names[y_set], 
                palette=['red', 'green', 'blue'], edgecolor="black", s=100)

plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())
plt.title(f"Clasificación KNN (k={k}) - Fronteras de Decisión")
plt.xlabel('Sépalo Largo (Estandarizado)')
plt.ylabel('Sépalo Ancho (Estandarizado)')
plt.legend()
plt.show()


In [ ]:
# TD_tree 
import numpy as np
import matplotlib.pyplot as plt

#INSERTION-SORT(A, n)
#1  for i = 2 to n
#2      key = A[i]
#3      // Insert A[i] into the sorted subarray A[1 : i - 1].
#4      j = i - 1
#5      while j > 0 and A[j] > key
#6          A[j + 1] = A[j]
#7          j = j - 1
#8      A[j + 1] = key
# interesante insercion!

class TD_Tree():
    def __init__(self,k):
        self.k = k
        self.arbol = None
    class Nodo:
        def __init__(self,punto):
            self.punto = punto 
            self.izq = None
            self.der = None
    #------------------------------------------------
    def establecer_niveles(self,n):
        niveles = np.log2(n + 1)               # 2^niveles - 1 = datos
        self.k = np.ceil(niveles).astype(int)   # redondeo hacia arriba, la parte entera
    #------------------------------------------------
    def distancia(self,p1,p2):
        d = 0
        n = len(p1)
        for i in range(n):
            d += (p1[i] - p2[i])**2
        return d**.5
    # -----------------------------------------------------
    def ordenar(self,puntos,eje):
        m, n = puntos.shape
        for i in range(1,m):
            p_compare = puntos[i].copy()          #se entiende por range no contiene a n, asi guarda es equivalente con los algoritmos
            v_compare = p_compare[eje]
            j = i-1
            while j>=0 and puntos[j,eje]>v_compare:
                puntos[j+1] = puntos[j]
                j = j - 1
            puntos[j+1] = p_compare
        return puntos
    #--------------------------------------------------------
    def construir(self,puntos,nivel=0):
        n = len(puntos)
        if n==0:
            return None
        dimension = len(puntos[0])      # (a,b,c) 3 dimensiones
        eje = nivel % dimension         # 0%3  1%3  2%3
        ordenados = self.ordenar(puntos,eje)
        ind_mediana = n//2
        p_mediana = puntos[ind_mediana]
        nodo = self.Nodo(p_mediana)
        nodo.izq = self.construir(ordenados[:ind_mediana],nivel + 1)
        nodo.der = self.construir(ordenados[ind_mediana + 1 :],nivel + 1)
        return nodo  
    #------------------------------------------------------------
    def buscar(self,nodo,p,nivel=0,mejor=None):
        if nodo is None:
            return mejor
        k = len(p)
        eje = nivel % k
        if p[eje]<=nodo.punto[eje]:
            lado_move = nodo.izq
            lado_otro = nodo.der
        else:
            lado_move = nodo.der
            lado_otro = nodo.izq
        mejor = self.buscar(lado_move,p,nivel+1)
        d = self.distancia(p,nodo.punto)
        if mejor is None or d < mejor[0]:
            mejor = (d,nodo.punto)
        d_hiperplano = abs(p[eje] - nodo.punto[eje])
        if d_hiperplano < mejor[0]:
            mejor = self.buscar(lado_otro,p,nivel + 1,mejor)
        return mejor
    #---------------------------------------------------------------
    def ordenar_reverse(self,d_vecinos):
        m = len(d_vecinos)
        for i in range(m-2,-1,-1):
            p_compare = d_vecinos[i]
            v_compare = p_compare[0]
            j = i + 1
            while j<m and d_vecinos[j][0]>v_compare:
                d_vecinos[j - 1] = d_vecinos[j]
                j = j + 1
            d_vecinos[j - 1] = p_compare
        return d_vecinos
    #----------------------------------------------------------------
    def buscar_k(self,nodo,p,k_vecinos,nivel=0,candidatos=None):   # ahora se buscan k vecinos
        if candidatos is None:
            candidatos = []
        if nodo is None:
            return candidatos
        k = len(p)
        eje = nivel % k
        if p[eje]<=nodo.punto[eje]:
            lado_move = nodo.izq
            lado_otro = nodo.der
        else:
            lado_move = nodo.der
            lado_otro = nodo.izq
        candidatos = self.buscar_k(lado_move,p,k_vecinos,nivel + 1,candidatos)
        d = self.distancia(p,nodo.punto)
        if len(candidatos)<k_vecinos:
            candidatos.append((d,nodo.punto))
            candidatos = self.ordenar_reverse(candidatos)
        elif d < candidatos[0][0]:
            candidatos[0] = (d,nodo.punto)
            candidatos = self.ordenar_reverse(candidatos)
        lleno = len(candidatos) == k_vecinos
        mayor_distancia = candidatos[0][0] if lleno else float('inf')
        d_hiperplano = abs(p[eje] - nodo.punto[eje])
        if  d_hiperplano < mayor_distancia:
            candidatos = self.buscar_k(lado_otro,p,k_vecinos,nivel+ 1,candidatos)
        return candidatos
    #---------------------------------------------------- 
    def buscar_vecino(self,arbol,p):   #(1.41, array([7,6]))
        candidatos = self.buscar(arbol,p) 
        vecinos_ = np.zeros((1,self.k)) 
        d = candidatos[0]
        vecinos_[0] = candidatos[1] 
        return vecinos_,d
    #----------------------------------------------------
    def buscar_k_vecinos(self,arbol,p,k):  # [(float,array),(float,array)..]
        candidatos = self.buscar_k(arbol,p,k)
        m = len(candidatos)
        vecinos_ = np.zeros((m,self.k))
        d = np.zeros((m,1))
        for i in range(m):
            d[i] = candidatos[i][0]
            vecinos_[i] = candidatos[i][1]
        return vecinos_,d
    #---------------------------------------------------
    def imprimir(self,nodo, nivel=0, rama="raiz"):
        if nodo is None:
            return
        k   = len(nodo.punto)
        eje = ["X","Y","Z","W"][nivel % k]
        print("  " * nivel + f"[{rama}] {nodo.punto}  (corte {eje}={nodo.punto[nivel%k]})")
        self.imprimir(nodo.izq, nivel + 1, "izq")
        self.imprimir(nodo.der, nivel + 1, "der")
    #-----------------------------------------------------------
    def imprimir_yo(self,nodo,nivel=0,rama="raiz"):
        if nodo is None:
            return
        k = len(nodo.punto)
        eje = ["X","Y","Z","W"][nivel % k]
        if rama == "raiz":
            print("\t"*self.k  + f"[{rama}] {nodo.punto} (corte {eje}={nodo.punto[nivel % k]})")
        if rama == "izq":
            print("\t"*(self.k-nivel) + f"[{rama}] {nodo.punto} (corte {eje}={nodo.punto[nivel % k]})")
        if rama == "der":
            print("\t"*self.k + f"[{rama}] {nodo.punto} (corte {eje}={nodo.punto[nivel % k]})"+"\t"*(self.k-nivel))
        self.imprimir_yo(nodo.izq,nivel+ 1,"izq")
        self.imprimir_yo(nodo.der,nivel+1,"der")
    #---------------------------------------
    def graficar(self,p_,vecinos,puntos):
        if self.k == 3:
            self.graficar_3d(p_,vecinos,puntos)
        else:
            plt.scatter(puntos[:,0],puntos[:,1],color='black') 
            plt.scatter(p_[:,0],p_[:,1],color='orange')
            plt.scatter(vecinos[:,0],vecinos[:,1], edgecolor='blue',label='vecino cercano')
            plt.legend()
            plt.show()
    #-----------------------------------------
    def graficar_3d(self, p_, vecinos, puntos):
        fig = plt.figure()
        ax  = fig.add_subplot(111, projection='3d')
        ax.scatter(puntos[:,0], puntos[:,1], puntos[:,2], color='black')
        ax.scatter(p_[:,0],     p_[:,1],     p_[:,2],     color='orange', marker='*', s=200)
        ax.scatter(vecinos[:,0],vecinos[:,1],vecinos[:,2], edgecolor='blue', facecolors='none', s=150)
        for v in vecinos:
            ax.plot([p_[0,0],v[0]], [p_[0,1],v[1]], [p_[0,2],v[2]], 'k--')
        plt.show()

#=====================================================
if __name__=='__main__':
    td_tree = TD_Tree(2)
    puntos_2d =np.array ([
        [2,8], [5,3], [7,6], [1,4], [9,2], [4,7], [6,1]
    ])
    arbol = td_tree.construir(puntos_2d)
    td_tree.imprimir(arbol)
    p = (6,5)
    p_ = np.array([p])
    vecinos,d = td_tree.buscar_k_vecinos(arbol,p,4)
    td_tree.graficar(p_,vecinos,puntos_2d)
    puntos_3d = np.array([
        [2,8,1], [5,3,7], [7,6,2], [1,4,9], [9,2,3],
        [4,7,5], [6,1,8], [3,5,4], [8,9,6], [2,3,7],
        [6,8,3], [1,6,2], [9,4,8], [5,7,1], [3,9,5]
    ])
    td_tree_3 = TD_Tree(3)
    arbol_3d = td_tree_3.construir(puntos_3d)
    td_tree_3.imprimir(arbol_3d)
    p = (6,5,4)
    p_ = np.array([p])
    vecinos,d = td_tree_3.buscar_k_vecinos(arbol_3d,p,3)
    td_tree_3.graficar(p_,vecinos,puntos_3d)
    print(vecinos)
    # ejercicio profe
    colores = ['Rojo', 'Verde', 'Azul', 'Amarillo', 'Negro', 'Blanco', 'Gris', 'Cyan', 'Magenta']
    colores_pixeles = np.array([
        [255, 0, 0],    # Rojo
        [0, 255, 0],    # Verde
        [0, 0, 255],    # Azul
        [255, 255, 0],  # Amarillo
        [0, 0, 0],      # Negro
        [255, 255, 255],# Blanco
        [128, 128, 128],# Gris
        [0, 255, 255],  # Cyan
        [255, 0, 255]   # Magenta
    ])
    p_s = np.array([
        [250, 10, 10],   # Casi Rojo puro
        [10, 10, 10],    # Casi Negro (Gris muy oscuro)
        [130, 130, 140], # Un gris azulado
        [200, 200, 50]   # Un amarillo oscuro/mostaza
    ])
    arbol_colores = td_tree_3.construir(colores_pixeles)
    td_tree_3.imprimir(arbol_colores)
    for e in p_s:
        vecinos, distancias = td_tree_3.buscar_k_vecinos(arbol_colores,e,1)
        e_ = np.array([e])
        td_tree_3.graficar(e_,vecinos,colores_pixeles)

In [ ]:
# SKLEARN
# SKLEARN
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KDTree

# ======================================================
# EJEMPLO 1: BÚSQUEDA ESPACIAL EN 2D
# ======================================================
print("--- EJEMPLO 1: Búsqueda de Vecinos con KD-Tree (2D) ---")

# 1. Generamos datos sintéticos (Puntos aleatorios)
np.random.seed(42)
points_2d = np.random.rand(40, 2)  # 40 puntos en un plano 2D (x, y)

# 2. Construcción del KD-Tree
# leaf_size=2: Obliga al árbol a dividir mucho el espacio (crea muchas ramas).
# metric='euclidean': KD-Tree funciona perfecto con distancia euclidiana.
tree = KDTree(points_2d, leaf_size=2, metric='euclidean')

# 3. Punto de consulta (Un punto nuevo al azar)
query_point = np.array([[0.5, 0.5]])

# 4. Consultar al árbol: Buscar los 3 vecinos más cercanos
k = 3
distancias, indices = tree.query(query_point, k=k)

print(f"Punto buscado: {query_point[0]}")
print(f"Índices de vecinos encontrados: {indices[0]}")
print(f"Distancias: {distancias[0]}")

# --- VISUALIZACIÓN ---
plt.figure(figsize=(8, 6))
# Todos los puntos (Gris)
plt.scatter(points_2d[:, 0], points_2d[:, 1], c='gray', label='Datos', alpha=0.6)
# Punto objetivo (Rojo)
plt.scatter(query_point[:, 0], query_point[:, 1], c='red', marker='*', s=300, label='Query Point')
# Vecinos encontrados (Círculos azules vacíos)
vecinos = points_2d[indices[0]]
plt.scatter(vecinos[:, 0], vecinos[:, 1], s=200, facecolors='none', edgecolors='blue', linewidths=2, label=f'{k} Vecinos más cercanos')

# Dibujar líneas conectando el punto con sus vecinos
for i in range(k):
    plt.plot([query_point[0,0], vecinos[i,0]], [query_point[0,1], vecinos[i,1]], 'k--', alpha=0.3)

plt.title(f'KD-Tree: Búsqueda de los {k} vecinos más cercanos')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()


# ======================================================
# EJEMPLO 2: APLICACIÓN REAL (Color Matching en 3D)
# ======================================================
print("\n--- EJEMPLO 2: 'Color Matching' usando Espacio RGB (3D) ---")
# El KD-Tree es excelente para buscar en 3 dimensiones (R, G, B).

# 1. Definimos una "Paleta de Colores" conocida (Datos de entrenamiento)
# Formato: [R, G, B] donde cada valor va de 0 a 255
palette_names = ['Rojo', 'Verde', 'Azul', 'Amarillo', 'Negro', 'Blanco', 'Gris', 'Cyan', 'Magenta']
palette_rgb = np.array([
    [255, 0, 0],    # Rojo
    [0, 255, 0],    # Verde
    [0, 0, 255],    # Azul
    [255, 255, 0],  # Amarillo
    [0, 0, 0],      # Negro
    [255, 255, 255],# Blanco
    [128, 128, 128],# Gris
    [0, 255, 255],  # Cyan
    [255, 0, 255]   # Magenta
])

# 2. Construimos el árbol con la paleta
color_tree = KDTree(palette_rgb, metric='euclidean')

# 3. Colores misteriosos (Inputs nuevos)
# Queremos saber a qué color de nuestra paleta se parecen más estos colores.
mystery_colors = np.array([
    [250, 10, 10],   # Casi Rojo puro
    [10, 10, 10],    # Casi Negro (Gris muy oscuro)
    [130, 130, 140], # Un gris azulado
    [200, 200, 50]   # Un amarillo oscuro/mostaza
])

# 4. Buscamos el vecino más cercano (k=1) para cada color misterioso
dists, idxs = color_tree.query(mystery_colors, k=1)

print("\nResultados del Color Matching:")
print(f"{'Color Input (RGB)':<20} -> {'Color Paleta Detectado':<15} (Distancia)")
print("-" * 60)

for i in range(len(mystery_colors)):
    input_c = mystery_colors[i]
    matched_idx = idxs[i][0]
    matched_name = palette_names[matched_idx]
    distance = dists[i][0]
    
    print(f"{str(input_c):<20} -> {matched_name:<15} ({distance:.2f})")

# Nota: Una distancia baja significa que el color es muy similar.

In [ ]:
# Ball - tree
import numpy as np 
import matplotlib.pyplot as plt

class Ball_Tree:
    def __init__(self,k,metrica=None):
        self.metrica = metrica
        self.k = k
    def distancia(self,a,b):
        if self.metrica == 'haversine':
            lat1 = a[0] ; lat2 = b[0]
            delta_latitud = a[0]- b[0]
            delta_longitud = a[1] - b[1]
            x = (np.sin(delta_latitud/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(delta_longitud/2)**2)
            return 2*6371*np.arcsin(np.sqrt(x))
        else:
            d = 0
            n = len(a)
            for i in range(n):
                d+=(a[i] - b[i])**2
            return d**0.5
#-----------------------------------------
    def mayor_distancia(self,c,puntos):  # [[,],[,]]
        d_max = -float('inf')
        for e in puntos:
            d = self.distancia(c,e)   #e = [,]
            if d > d_max:
                d_max = d
        return d_max
#-------------------------------------------
    def varianza(self,A):
        c = A.sum(axis=0)/len(A)
        var = []
        for i in range(len(A[0])):
            var.append(sum((A[:,i]-c[i])**2)/len(A))
        return var
#----------------------------------------------
    def ind_maximo(self,array):
        ind_mayor = -1
        mayor = -float('inf')
        for i in range(len(array)):
            if array[i] > mayor:
                ind_mayor = i 
                mayor = array[i]
        ejes = ["X","Y","Z"]
        return ejes[ind_mayor],ind_mayor
#---------------------------------------------
    def ordenar(self,A_, eje):
        n = len(A_)
        for i in range(1, n):
            p     = A_[i].copy()
            valor = A_[i, eje]
            j     = i - 1
            while j >= 0 and A_[j, eje] > valor:
                A_[j+1] = A_[j]
                j = j - 1
            A_[j+1] = p
        return A_
#------------------------------------
    def dividir_eje_mediana(self,A, eje):
        A_eje      = A.copy()
        A_ordenado = self.ordenar(A_eje, eje)
        Me         = len(A_ordenado) // 2
        A_izq      = A_ordenado[:Me].copy()
        A_der      = A_ordenado[Me:].copy()
        return A_izq, A_der
#-------------------------------------
    class Nodo:
        def __init__(self, centroide, radio, punto=None):
            self.centroide = centroide   # centro de la bola
            self.radio     = radio       # radio de la bola
            self.punto     = punto       # solo si es hoja
            self.izq       = None
            self.der       = None
#------------------------------------------------
    def construir(self,puntos):
        n          = len(puntos)
        c          = puntos.sum(axis=0) / n
        r          = self.mayor_distancia(c, puntos)
        # caso base: hoja
        if n == 1:
            return self.Nodo(c, r, punto=puntos[0])
        # dividir por eje de mayor varianza
        eje_nombre, ind_eje = self.ind_maximo(self.varianza(puntos))
        A_izq, A_der        = self.dividir_eje_mediana(puntos, ind_eje)
        nodo      = self.Nodo(c, r)
        nodo.izq  = self.construir(A_izq)
        nodo.der  = self.construir(A_der)
        return nodo
#------------------------------------------------------
    def ordenar_reverse(self,d_vecinos):
        m = len(d_vecinos)
        for i in range(m-2, -1, -1):
            p_compare = d_vecinos[i]
            v_compare = p_compare[0]
            j         = i + 1
            while j < m and d_vecinos[j][0] > v_compare:
                d_vecinos[j-1] = d_vecinos[j]
                j               = j + 1
            d_vecinos[j-1] = p_compare
        return d_vecinos
#----------------------------------------------
    def buscar_k(self,nodo, consulta, k_vecinos, candidatos=None):
        if candidatos is None:
            candidatos = []
        if nodo is None:
            return candidatos
        d_centroide = self.distancia(consulta, nodo.centroide)
        lleno = len(candidatos) == k_vecinos
        peor_distancia  = candidatos[0][0] if lleno else float('inf')
        minima_posible  = d_centroide - nodo.radio
        if lleno and minima_posible >= peor_distancia:
            return candidatos   # PODANDO toda la bola
        if nodo.punto is not None:  # caso hoja: evaluar el punto directamente
            d = self.distancia(consulta, nodo.punto)
            if len(candidatos) < k_vecinos:
                candidatos.append((d, nodo.punto))
                candidatos = self.ordenar_reverse(candidatos)
            elif d < candidatos[0][0]:
                candidatos[0] = (d, nodo.punto)
                candidatos    = self.ordenar_reverse(candidatos)
            return candidatos
        d_izq = self.distancia(consulta, nodo.izq.centroide)  # explorando primero el hijo cuyo centroide esta mas cerca
        d_der = self.distancia(consulta, nodo.der.centroide)
        if d_izq <= d_der:
            candidatos = self.buscar_k(nodo.izq, consulta, k_vecinos, candidatos)
            candidatos = self.buscar_k(nodo.der, consulta, k_vecinos, candidatos)
        else:
            candidatos = self.buscar_k(nodo.der, consulta, k_vecinos, candidatos)
            candidatos = self.buscar_k(nodo.izq, consulta, k_vecinos, candidatos)
        return candidatos
#-------------------------------------------------
    def imprimir(self,nodo, nivel=0, rama="raiz"):
        if nodo is None:
            return
        prefijo = "  " * nivel
        if nodo.punto is not None:
            print(prefijo + f"[{rama}] HOJA punto={nodo.punto}  c={np.round(nodo.centroide,2)}  r={nodo.radio:.4f}")
        else:
            print(prefijo + f"[{rama}] c={np.round(nodo.centroide,2)}  r={nodo.radio:.4f}")
        self.imprimir(nodo.izq, nivel+1, "izq")
        self.imprimir(nodo.der, nivel+1, "der")
#------------------------------------------------
    def buscar_k_vecinos(self,arbol,p,k):
        candidatos = self.buscar_k(arbol,p,k)  # [(float,[]),(float,[]),()]
        vecinos = []
        distancias = []
        for i in range(len(candidatos)):
            distancias.append(candidatos[i][0])
            vecinos.append(candidatos[i][1])
        distancias = np.array(distancias)
        vecinos = np.array(vecinos)
        return vecinos,distancias
#-------------------------------------------------
    def graficar(self,arbol,p,vecinos):
        plt.figure()
        plt.scatter(arbol[:,0],arbol[:,1],color='black',alpha=0.5)
        plt.scatter(vecinos[:,0],vecinos[:,1],edgecolor='blue',facecolors='none')
        plt.scatter(p[:,0],p[:,1],marker='*')
        for  v in vecinos:
            plt.plot([p[0,0],v[0]],[p[0,1],v[1]],'k--')
        plt.show()
#-----------------------------------------------
    def distancia_esferica(a,b):
        return 
if __name__=='__main__':
    #A = np.array([[2,8],[5,3],[7,6],[1,4],[9,2],[4,7],[6,1]])
    rng = np.random.RandomState(42)
    A = rng.random_sample((50, 2))
    ball_tree = Ball_Tree(len(A[0]))
    arbol = ball_tree.construir(A)
    ball_tree.imprimir(arbol)
    p = np.array([[0.5],[0.5]])
    p_ = p.T
    vecinos, distancias = ball_tree.buscar_k_vecinos(arbol,p,5)
    ball_tree.graficar(A,p_,vecinos)
    ball_tree_geo = Ball_Tree(2,metrica='haversine')
    cities = {
    "CDMX": [19.4326, -99.1332],
    "Buenos Aires": [-34.6037, -58.3816],
    "Madrid": [40.4168, -3.7038],
    "Tokio": [35.6762, 139.6503],
    "Nueva York": [40.7128, -74.0060],
    "Lima": [-12.0464, -77.0428]
    }
    city_names = list(cities.keys())
    coords_deg = np.array(list(cities.values()))
    A = np.radians(coords_deg)
    arbol = ball_tree_geo.construir(A)
    bogota_deg = [4.7110, -74.0721]
    p = np.radians([[bogota_deg[0]], [bogota_deg[1]]])  # shape (2,1)  igual que antes
    p_ = p.T
    vecinos, distancias = ball_tree_geo.buscar_k_vecinos(arbol,p,2)
    ball_tree_geo.graficar(A,p_,vecinos)

In [ ]:
# SKLEARN
# SKLEARN , aunque es..
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import BallTree

# ======================================================
# EJEMPLO 1: BÚSQUEDA VISUAL EN 2D (Euclidiana)
# ======================================================
print("--- EJEMPLO 1: Búsqueda de Vecinos en 2D ---") 
rng = np.random.RandomState(42)
X = rng.random_sample((50, 2))  # Puntos entre [0, 1]

# 2. Construimos el Ball Tree
# leaf_size: Determina cuándo dejar de dividir nodos. 
# Afecta la velocidad de construcción vs consulta.
tree = BallTree(X, leaf_size=15, metric='euclidean')
# 3. Punto de consulta (¿Quiénes son los vecinos de este punto?)
p = [[0.5, 0.5]]
# 4. Buscamos los 5 vecinos más cercanos (k=5)
# retorna distancias y los índices de los puntos en X
dist, ind = tree.query(p, k=5)

print(f"Índices de los 5 vecinos más cercanos: {ind[0]}")
print(f"Distancias a los vecinos: {dist[0]}")

# --- VISUALIZACIÓN ---
plt.figure(figsize=(8, 6))
# Dibujar todos los puntos (Gris)
plt.scatter(X[:, 0], X[:, 1], c='gray', alpha=0.5, label='Datos')
# Dibujar el punto de consulta (Rojo)
plt.scatter(p[0][0], p[0][1], c='red', marker='*', s=200, label='Punto Consulta')
# Dibujar los vecinos encontrados (Círculos Azules grandes)
plt.scatter(X[ind[0], 0], X[ind[0], 1], s=150, facecolors='none', edgecolors='blue', linewidths=2, label='Vecinos (k=5)')

# Dibujar líneas conectando
for i in ind[0]:
    plt.plot([p[0][0], X[i, 0]], [p[0][1], X[i, 1]], 'k--', alpha=0.3)

plt.title('Búsqueda con Ball Tree (Euclidiana)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


# ======================================================
# EJEMPLO 2: EL SUPERPODER (Datos Geoespaciales / Haversine)
# ======================================================
print("\n--- EJEMPLO 2: Búsqueda Geoespacial (Tierra) ---")
# KD-Tree no funciona bien aquí porque la Tierra es curva. Ball Tree sí.

# Coordenadas: [Latitud, Longitud] (en grados)
# Ciudad de México, Buenos Aires, Madrid, Tokio, Nueva York
cities = {
    "CDMX": [19.4326, -99.1332],
    "Buenos Aires": [-34.6037, -58.3816],
    "Madrid": [40.4168, -3.7038],
    "Tokio": [35.6762, 139.6503],
    "Nueva York": [40.7128, -74.0060],
    "Lima": [-12.0464, -77.0428]
}

# Convertir diccionario a array y PASAR A RADIANES (Requisito de sklearn para Haversine)
city_names = list(cities.keys())
coords_deg = np.array(list(cities.values()))
coords_rad = np.radians(coords_deg)

# Construir Ball Tree con métrica 'haversine'
# La métrica haversine asume una esfera unitaria.
geo_tree = BallTree(coords_rad, metric='haversine')

# Punto de consulta: Bogotá, Colombia
bogota_deg = [4.7110, -74.0721]
bogota_rad = np.radians([bogota_deg])

# Buscar las 2 ciudades más cercanas a Bogotá en nuestra lista
k_geo = 2
dist_rad, ind_geo = geo_tree.query(bogota_rad, k=k_geo)

# Convertir distancia de radianes a kilómetros
# Radio de la Tierra aprox = 6371 km
earth_radius_km = 6371
dist_km = dist_rad[0] * earth_radius_km

print(f"Ciudades más cercanas a Bogotá ({bogota_deg}):")
for i, idx in enumerate(ind_geo[0]):
    name = city_names[idx]
    d_km = dist_km[i]
    print(f"{i+1}. {name} -> a {d_km:.2f} km")